### End-to-End 지연 분해
- 파일 : n2-4_e2e.ipynb
- 역할 : 전처리부터 후처리까지 단계별 시간을 분해해 병목을 찾습니다.
- 기능 :
    1. 단계별 지연 측정  
    2. 원본 해상도별 전처리 시간 측정  
    3. 프레임 샘플링 적용  


>> [준비] 실습 저장소 받기 (노트북 전용 셀)
<hr>

In [ ]:
#- 실습 저장소 받기: 노트북마다 런타임이 분리되므로 새 세션에서 한 번 실행
import os
if not os.path.exists("npu-course"):
    !git clone -q https://github.com/npu-ondevice-ai/npu-course.git
print("준비 완료:", os.path.exists("npu-course/data"))

# 확인 포인트 ------------------------------------------
# - 준비 완료: True 출력 체크 (이 노트북은 /content 위치에서 실행)

>> [준비 2] 패키지·측정 함수·ONNX 준비 (노트북 전용 셀)

In [ ]:
#- 이 노트북이 이어받는 것들: 패키지, measure(), mobilenet_v2.onnx
!pip install -q onnx onnxruntime "protobuf<6"

#- 모듈 로딩
import os
import time
import numpy as np
import torch
import onnxruntime as ort
from torchvision import models

def measure(fn, n_warmup=10, n_runs=100):
    for _ in range(n_warmup):
        fn()
    t = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        t.append((time.perf_counter() - t0) * 1000)
    return np.array(t)

if not os.path.exists("mobilenet_v2.onnx"):
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1).eval()
    torch.onnx.export(m, torch.randn(1, 3, 224, 224), "mobilenet_v2.onnx",
                      opset_version=13, input_names=["input"],
                      output_names=["output"], dynamo=False)
print("준비 완료:", os.path.exists("mobilenet_v2.onnx"))

# 확인 포인트 ------------------------------------------
# - 준비 완료: True 출력 체크

>> [코드 6-5] 단계별 지연 측정 (n2-4_e2e.ipynb, STEP 1)
<hr>

In [ ]:
#- 모듈로딩
import cv2
from collections import defaultdict

#- 전처리 상수·실행기·입력 영상
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
sess = ort.InferenceSession("mobilenet_v2.onnx", providers=["CPUExecutionProvider"])
cap = cv2.VideoCapture("npu-course/data/sample_video.mp4")
T = defaultdict(list)

#- 프레임마다 네 단계의 시간을 따로 기록
while True:
    t0 = time.perf_counter()
    ok, frame = cap.read()                                               # ① 디코드
    if not ok: break
    t1 = time.perf_counter()
    img = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224)) # ② 전처리

    # 정규화와 NCHW 변환 (전처리의 나머지 절반)
    x = ((img.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)[None]
    t2 = time.perf_counter()
    logits = sess.run(None, {"input": x})[0]                            # ③ 추론
    t3 = time.perf_counter()
    top1 = int(logits.argmax())                                         # ④ 후처리
    t4 = time.perf_counter()
    kab_list = [("decode", t0, t1), ("pre", t1, t2), ("infer", t2, t3), ("post", t3, t4)]
    for k, a, b in kab_list: T[k].append((b - a) * 1000)

#- 단계별 평균
for k, v in T.items():
    print(f"{k:7s} {np.mean(v):7.2f} ms")

# 확인 포인트 -------------------------------------------------------------------------
# - 네 단계의 평균과 전체 대비 비율 계산 확인

>> [코드 6-6] 원본 해상도별 전처리 시간 측정 (STEP 2)
<hr>

In [ ]:
#- 세 해상도의 가짜 프레임으로 전처리 시간만 측정
for w, h in [(1920, 1080), (1280, 720), (640, 480)]:
    frame = np.random.randint(0, 256, (h, w, 3), dtype=np.uint8)
    t = measure(lambda: cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224)))
    print(f"{w}x{h}: 평균 {t.mean():.2f} ms")

# 확인 포인트 -------------------------------------------------------------------------
# - 해상도 감소에 따른 시간 감소 확인, 픽셀 수 비율과 비교

>> [코드 6-7] 프레임 샘플링 적용 (STEP 3)
<hr>

In [ ]:
#- 샘플링 주기
K = 3                                          # 3프레임마다 한 번 추론

#- K프레임마다만 추론, 나머지는 직전 결과 재사용
cap = cv2.VideoCapture("npu-course/data/sample_video.mp4")
i, n_infer, last = 0, 0, None

while True:
    ok, frame = cap.read()
    if not ok: break
    # K 프레임마다 추론
    if i % K == 0:
        img = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224))
        x = ((img.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)[None]
        last = int(sess.run(None, {"input": x})[0].argmax()); n_infer += 1
    i += 1                                     # 나머지 프레임은 직전 결과 사용
print(f"전체 {i}프레임 중 {n_infer}프레임 추론")

# 확인 포인트 --------------------------------------------------------------------------
# - 전체 프레임 수와 추론 프레임 수(≈ 전체 / K) 출력 체크